# Geração de patches e manifesto de dados (estágio 06)

Divide o composite multiespectral normalizado (estágio 05) e a máscara binária final (estágio 04) em patches de `data.patch_size` (512x512 px) sobre um grid determinístico sem sobreposição, filtra os patches pela proporção mínima de café (`data.coffee_min_ratio`) e registra cada um em `MyDrive/tcc/data/processed/manifest.parquet` (paths relativos à raiz `tcc/`). A geração é idempotente: execuções repetidas reutilizam o manifesto vigente e os patches já persistidos, preservando o processamento. Cada batch de patches é versionado por um fingerprint das entradas (composite + máscara + configuração), de modo que mudanças nas entradas geram uma subpasta nova sem sobrescrever a anterior. Os patches e o manifesto são as entradas dos estágios 07 (k-fold espacial), 08 (EDA) e 09/10 (treino).

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/oguel/tcc-umamba/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt` (incluindo rasterio, usado na leitura dos GeoTIFFs, e pandas/pyarrow, usados na escrita do manifesto), garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)

## Atualização dos módulos `src/` em memória

Remove do cache do kernel (`sys.modules`) os módulos `src.*` carregados em execuções anteriores, garantindo que as próximas importações usem a versão recém-sincronizada pelo bootstrap (evita módulos obsoletos após edições do código).

In [ ]:
# Remove os módulos src.* em cache para forçar o carregamento da versão atual do workspace.
import sys

for module in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[module]
print("Módulos src.* recarregados do workspace sincronizado.")

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`, incluindo a subpasta de patches e o manifesto.

In [ ]:
# Importa o pacote compartilhado, identifica a plataforma e resolve os caminhos de armazenamento.
from src import io
from src.config import get_config

platform = io.detect_platform()
storage_paths = io.resolve_storage_paths()
config = get_config()
print(f"Plataforma: {platform}")
print(f"Patches: {storage_paths['data_processed_patches']}")
print(f"Manifesto: {storage_paths['data_processed'] / 'manifest.parquet'}")

## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, garantindo o mesmo protocolo de execução nas duas plataformas.

In [ ]:
# Fixa sementes e flags determinísticas do PyTorch de acordo com a configuração.
from src.utils import set_all_seeds, set_deterministic_flags

set_all_seeds(config["reproducibility"]["seed"])
set_deterministic_flags()
print(f"Seed fixada: {config['reproducibility']['seed']}")

## Dependências dos estágios 05 e 04

Verifica que o composite normalizado do estágio 05 e a máscara binária final do estágio 04 estão disponíveis no caminho canônico — sem estas dependências a geração de patches não pode prosseguir.

In [ ]:
# Verifica as dependências do estágio 05 (composite) e do estágio 04 (máscara final).
from src.data.mask_finalization import final_mask_path
from src.data.preprocessing import composite_path

dependencies = {
    "Composite normalizado (estágio 05)": composite_path(storage_paths),
    "Máscara binária final (estágio 04)": final_mask_path(storage_paths),
}
for name, path in dependencies.items():
    if not io.path_exists(path):
        raise FileNotFoundError(f"Dependência não encontrada: {path}")
    print(f"Disponível: {name}: {path}")

## Geração de patches

Garante (idempotente) os patches 512x512 e o manifesto em `MyDrive/tcc/data/processed/`: o composite do estágio 05 é cortado no grid da máscara do estágio 04 (mesmo grid, sem sobreposição, bordas parciais descartadas) e cada patch é persistido em uma subpasta versionada pelo fingerprint das entradas (`images/<hash>/<patch_id>.npy` e `masks/<hash>/<patch_id>.npy`), filtrado pela proporção mínima de café. Manifesto vigente é reutilizado; quando as entradas mudam, os patches novos vão para uma subpasta nova, preservando os anteriores.

In [ ]:
# Garante os patches e o manifesto (reutiliza o que já existe).
from src.data.patch_generation import generate_patches

manifest_file = generate_patches(storage_paths)

## Verificação do manifesto

Confere o manifesto persistido: total de patches, distribuição da proporção de café, formato dos arrays e fontes de máscara registradas.

In [ ]:
# Verifica o manifesto persistido (contagem, café e formato dos patches).
from src.data.patch_generation import verify_manifest

manifest_stats = verify_manifest(storage_paths)
coffee = manifest_stats["coffee_ratio"]
print(f"Patches registrados: {manifest_stats['n_patches']} (com café: {manifest_stats['n_with_coffee']})")
print(f"Proporção de café: [mín {coffee['min']:.3f}, média {coffee['mean']:.3f}, máx {coffee['max']:.3f}]")
print(f"Formato dos patches: {manifest_stats['patch_shape']}")
print(f"Fontes de máscara: {manifest_stats['mask_source']}")

## Mosaico de amostras

Renderiza e persiste um mosaico com as amostras mais cafeeiras (RGB do composite + máscara binária) em `MyDrive/tcc/artifacts/figures/`; execuções repetidas reutilizam a figura já existente (idempotência).

In [ ]:
# Renderiza e persiste a figura do mosaico de amostras (reutiliza se já existir).
from src.data.patch_generation import save_patch_montage

montage_path = save_patch_montage(storage_paths)

## Resumo da etapa

Exibe o resumo da geração de patches: dependências reutilizadas, manifesto persistido, estatísticas de verificação e figura de registro.

In [ ]:
# Exibe o resumo da etapa de geração de patches.
summary = {
    "Composite de entrada (estágio 05)": str(dependencies["Composite normalizado (estágio 05)"]),
    "Máscara de referência (estágio 04)": str(dependencies["Máscara binária final (estágio 04)"]),
    "Manifesto": str(manifest_file),
    "Patches registrados": manifest_stats["n_patches"],
    "Patches com café": manifest_stats["n_with_coffee"],
    "Formato dos patches": manifest_stats["patch_shape"],
    "Figura de amostras": str(montage_path),
}
for key, value in summary.items():
    print(f"{key}: {value}")
print("Estágio 06 concluído.")